# ==============================================================================
# 🚢 MARITIME EVACUATION SURVIVAL: TITANIC VIA BAGGING CLASSIFIER
# ==============================================================================
### Core Theoretical Principles:
Bootstrap Aggregation (Breiman, 1996) draws $M$ independent bootstrap samples with replacement:
$$\hat{y}_{\text{bag}}(x) = \text{mode}\Big\{ h_1(x), h_2(x), \dots, h_M(x) \Big\}$$

Roughly $1 - 1/e \approx 63.2\%$ of samples are selected per tree, leaving $\approx 36.8\%$ as **Out-Of-Bag (OOB)** samples providing free internal cross-validation!


In [1]:
# STEP 1: ENTERPRISE ENVIRONMENT & REPRODUCIBILITY SEED
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Enterprise Bagging Environment initialized.")


✅ STEP 1: Enterprise Bagging Environment initialized.


## 📥 STEP 2 & 3: RAW DATASET INGESTION & 3-SECOND HEALTH AUDIT
Loading Titanic passenger manifest (891 passenger records).


In [2]:
# STEP 2 & 3: INGESTION & AUDIT
data_path = 'data/titanic/Titanic-Dataset.csv'
df = pd.read_csv(data_path)

print(f"📊 Dataset Dimensions: {df.shape[0]} passengers, {df.shape[1]} attributes")
print(f"Survival Distribution:\n{df['Survived'].value_counts(normalize=True).round(3)}")
df.head(3)


📊 Dataset Dimensions: 891 passengers, 12 attributes
Survival Distribution:
Survived
0    0.616
1    0.384
Name: proportion, dtype: float64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## 🧹 STEP 4 & 5: STRICT SEGREGATION & HYGIENE
Separating features and target `Survived`.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
y = df['Survived'].copy()

print(f"Features: {X.columns.tolist()}")


Features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


## 🔒 STEP 6: ZERO-DATA-LEAKAGE STRATIFIED SPLIT
Stratified 80/20 train/test split.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train Cohort: {X_train.shape[0]} passengers")
print(f"Test Cohort : {X_test.shape[0]} passengers")


Train Cohort: 712 passengers
Test Cohort : 179 passengers


## ⚙️ STEP 7: PREPROCESSOR PIPELINE
Standardizing numerical indicators and encoding categorical attributes.


In [5]:
# STEP 7: PREPROCESSOR PIPELINE
num_cols = ['Age', 'Fare', 'Pclass', 'SibSp', 'Parch']
cat_cols = ['Sex', 'Embarked']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])
print("✅ STEP 7: Preprocessor ready.")


✅ STEP 7: Preprocessor ready.


## ⚠️ STEP 8: BASELINE DUEL (SINGLE TREE VS BAGGING ENSEMBLE)
Contrasting a high-variance single decision tree against an ensemble of 50 bagged trees.


In [6]:
# STEP 8: BASELINE DUEL
pipe_single = Pipeline([
    ('prep', preprocessor),
    ('dt', DecisionTreeClassifier(random_state=42))
]).fit(X_train, y_train)

pipe_bag = Pipeline([
    ('prep', preprocessor),
    ('bag', BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=50,
        oob_score=True,
        random_state=42
    ))
]).fit(X_train, y_train)

acc_single = pipe_single.score(X_test, y_test)
acc_bag = pipe_bag.score(X_test, y_test)
oob_score = pipe_bag.named_steps['bag'].oob_score_

print("="*65)
print(f"Single Decision Tree Accuracy : {acc_single * 100:.2f}%")
print(f"Bagging (50 Trees) Accuracy   : {acc_bag * 100:.2f}%")
print(f"Out-Of-Bag (OOB) Internal Acc : {oob_score * 100:.2f}%")
print("="*65)


Single Decision Tree Accuracy : 81.56%
Bagging (50 Trees) Accuracy   : 81.01%
Out-Of-Bag (OOB) Internal Acc : 79.92%


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing `n_estimators` and `max_samples`.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'bag__n_estimators': [50, 100, 150],
    'bag__max_samples': [0.7, 0.85, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_bag, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Bagging Parameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV ROC-AUC: {grid.best_score_*100:.2f}%")


🏆 Best Bagging Parameters: {'bag__max_samples': 0.7, 'bag__n_estimators': 150}
🎯 Best 5-Fold CV ROC-AUC: 86.55%


## 📊 STEP 10 & 11: EVALUATION REPORT & DIAGNOSTICS
Comprehensive classification report and confusion matrix.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("📋 MARITIME SAFETY BAGGING REPORT:")
print(classification_report(y_test, y_pred, target_names=['Perished', 'Survived']))
print(f"Test ROC-AUC Score : {roc_auc_score(y_test, y_prob):.4f}")
print(f"Test Set Accuracy  : {accuracy_score(y_test, y_pred)*100:.2f}%")


📋 MARITIME SAFETY BAGGING REPORT:
              precision    recall  f1-score   support

    Perished       0.81      0.85      0.83       110
    Survived       0.75      0.68      0.71        69

    accuracy                           0.79       179
   macro avg       0.78      0.77      0.77       179
weighted avg       0.79      0.79      0.79       179

Test ROC-AUC Score : 0.8176
Test Set Accuracy  : 78.77%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & SMOKE TEST
Deploying pipeline and validating ensemble latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/titanic_bagging_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_passenger = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_class = loaded_pipe.predict(sample_passenger)[0]
pred_prob = loaded_pipe.predict_proba(sample_passenger)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🚢 LIVE PASSENGER BAGGED INFERENCE:")
print(f"   Outcome: {'✅ SURVIVED' if pred_class == 1 else '❌ PERISHED'}")
print(f"   Bagged Probability  : {pred_prob * 100:.2f}%")
print(f"   Inference Latency   : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/titanic_bagging_pipeline.joblib (385,993 bytes)

🚢 LIVE PASSENGER BAGGED INFERENCE:
   Outcome: ❌ PERISHED
   Bagged Probability  : 14.00%
   Inference Latency   : 31.561 ms (SLA < 2.0ms: CHECK)
